In this notebook, we'll perform feature engineering and data preparation. We'll clean, merge, and enrich the datasets with population, surface area, and vehicle fleet data to create the final indicators for the subsequent analysis.

In [1]:
import pandas as pd

In [2]:
df_incidents = pd.read_csv("data/traffic_incidents.csv")
df_municipalities = pd.read_csv("data/municipalities_metrics.csv")
df_aci = pd.read_csv("data/aci_fleet.csv")

In [3]:
#a quick check 
df_incidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 257220 entries, 0 to 257219
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   REF_AREA               257220 non-null  int64  
 1   DATA_TYPE              257220 non-null  str    
 2   RESULT                 257220 non-null  str    
 3   TIME_PERIOD            257220 non-null  int64  
 4   OBS_VALUE              257220 non-null  int64  
 5   Codice_Comune          257220 non-null  float64
 6   Comune                 257187 non-null  str    
 7   Superficie(Kmq)        257220 non-null  float64
 8   Popolazione_Residente  257220 non-null  float64
 9   Anno_riferimento       257220 non-null  float64
 10  Regione                257220 non-null  str    
dtypes: float64(4), int64(3), str(4)
memory usage: 21.6 MB


In [4]:
df_municipalities.info()

<class 'pandas.DataFrame'>
RangeIndex: 87454 entries, 0 to 87453
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Codice_Comune          87454 non-null  int64  
 1   Comune                 87443 non-null  str    
 2   Superficie(Kmq)        87454 non-null  float64
 3   Popolazione_Residente  87454 non-null  int64  
 4   Anno_riferimento       87454 non-null  int64  
 5   Regione                87454 non-null  str    
dtypes: float64(1), int64(3), str(2)
memory usage: 4.0 MB


In [6]:
df_aci.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Regione         20 non-null     str  
 1   Avg_Parco_Auto  20 non-null     int64
dtypes: int64(1), str(1)
memory usage: 452.0 bytes


We can now proceed with further dataset manipulation, trying to create three distinct columns for the "DATA_TYPE" column

In [8]:
df_acc = df_incidents[(df_incidents["DATA_TYPE"] == "ROADACC") & (df_incidents["RESULT"] == "9")][["REF_AREA","TIME_PERIOD", "OBS_VALUE", "Comune", "Superficie(Kmq)","Popolazione_Residente","Regione"]].rename(columns={"OBS_VALUE":"Incidenti"}).reset_index(drop=True)
df_inj = df_incidents[(df_incidents["DATA_TYPE"] == "KILLINJ") & (df_incidents["RESULT"] == "F")][["REF_AREA","TIME_PERIOD", "OBS_VALUE", "Comune", "Superficie(Kmq)","Popolazione_Residente","Regione"]].rename(columns={"OBS_VALUE":"Feriti"}).reset_index(drop=True)
df_dead = df_incidents[(df_incidents["DATA_TYPE"] == "KILLINJ") & (df_incidents["RESULT"] == "M")][["REF_AREA","TIME_PERIOD", "OBS_VALUE", "Comune", "Superficie(Kmq)","Popolazione_Residente","Regione"]].rename(columns={"OBS_VALUE":"Morti"}).reset_index(drop=True)

In [10]:
df_merged = df_acc.merge(df_inj, on=["REF_AREA","TIME_PERIOD"], how="outer") # outer join to ensure no loss in data points
df_merged = df_merged.merge(df_dead, on=["REF_AREA","TIME_PERIOD"], how="outer")
df_merged = df_merged[["Comune","Regione", "TIME_PERIOD", "Incidenti", "Feriti", "Morti","Superficie(Kmq)","Popolazione_Residente"]]
df_merged = df_merged.rename(columns={
    "REF_AREA" : "Codice_Comune",
    "TIME_PERIOD" : "Anno"
})

In [11]:
df_merged

,Comune,Regione,Anno,Incidenti,Feriti,Morti,Superficie(Kmq),Popolazione_Residente
0,Agliè,Piemonte,2014,6,11,0,13.1462,2701.0
1,Agliè,Piemonte,2015,5,9,0,13.1462,2644.0
2,Agliè,Piemonte,2016,5,8,0,13.1462,2661.0
3,Agliè,Piemonte,2017,0,0,0,13.1462,2658.0
4,Agliè,Piemonte,2018,1,1,0,13.1462,2634.0
...,...,...,...,...,...,...,...,...
85735,Villaspeciosa,Sardegna,2020,2,3,0,27.1937,2549.0
85736,Villaspeciosa,Sardegna,2021,5,7,0,27.1943,2536.0
85737,Villaspeciosa,Sardegna,2022,1,1,0,27.1943,2575.0
85738,Villaspeciosa,Sardegna,2023,4,5,0,27.1943,2616.0


In [12]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 85740 entries, 0 to 85739
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Comune                 85729 non-null  str    
 1   Regione                85740 non-null  str    
 2   Anno                   85740 non-null  int64  
 3   Incidenti              85740 non-null  int64  
 4   Feriti                 85740 non-null  int64  
 5   Morti                  85740 non-null  int64  
 6   Superficie(Kmq)        85740 non-null  float64
 7   Popolazione_Residente  85740 non-null  float64
dtypes: float64(2), int64(4), str(2)
memory usage: 5.2 MB


In [15]:
#we can now enrich the dataset, computing the accident fatality index and injury index
df_merged["Indice Fatalità"] = (df_merged["Morti"]/df_merged["Incidenti"]).fillna(0).round(2)
df_merged["Indice Lesività"] = (df_merged["Feriti"]/df_merged["Incidenti"]).fillna(0).round(2)


In [24]:
df_historical = df_merged.copy()
df_historical["Comune"] = df_historical["Comune"].fillna("None") #same issue with None municipality
df_historical.to_csv("data/historical_accidents.csv", index=False)

To analyze the data at the finest level of granularity, we will group the dataset by municipality (Comune) and calculate the mean of the various metrics across the time period. The exact same aggregation and averaging process will then be applied at the regional level (Regione) to analyze broader spatial trends.

In [36]:
df_municipalities_aggregated = df_merged.groupby(["Comune", "Regione"])[["Incidenti","Feriti", "Morti","Superficie(Kmq)","Popolazione_Residente"]].mean().reset_index()
df_municipalities_aggregated = df_municipalities_aggregated.rename(columns={
    "Incidenti":"Media Incidenti",
    "Feriti":"Media Feriti",
    "Morti":"Media Morti",
    "Superficie(Kmq)":"Media Superficie(Kmq)",
    "Popolazione_Residente":"Media Popolazione"
})

In [37]:
df_municipalities_aggregated


,Comune,Regione,Media Incidenti,Media Feriti,Media Morti,Media Superficie(Kmq),Media Popolazione
0,Abano Terme,Veneto,53.000000,64.181818,0.818182,21.408100,19982.000000
1,Abbadia Cerreto,Lombardia,0.000000,0.000000,0.000000,6.198600,283.800000
2,Abbadia Lariana,Lombardia,27.818182,39.181818,0.363636,16.644800,3199.545455
3,Abbadia San Salvatore,Toscana,8.636364,13.000000,0.181818,58.993300,6227.545455
4,Abbasanta,Sardegna,3.818182,5.181818,0.181818,39.846827,2632.272727
...,...,...,...,...,...,...,...
8164,Zuglio,Friuli-Venezia Giulia,0.400000,0.500000,0.000000,18.211300,567.800000
8165,Zumaglia,Piemonte,0.500000,0.600000,0.000000,2.613670,1015.500000
8166,Zumpano,Calabria,2.636364,5.090909,0.090909,8.083800,2600.545455
8167,Zungoli,Campania,0.000000,0.000000,0.000000,19.214200,1029.800000


In [39]:
#We can now compute some metrics to enhance our future analysis
df_municipality_indicators = df_municipalities_aggregated.copy()


In [41]:
#Accidents per 1,000 inhabitants (Per capita accident rate)
df_municipality_indicators["Incidenti_Pro_Capite_1000"] = ((df_municipality_indicators["Media Incidenti"]/df_municipality_indicators["Media Popolazione"])*1000).fillna(0)
#Total casualties (Fatalities + Injured) per 1,000 inhabitants (Per capita casualty rate)
df_municipality_indicators["Lesività_Pro_Capite_1000"] = (((df_municipality_indicators["Media Feriti"] + df_municipality_indicators["Media Morti"]) /df_municipality_indicators["Media Popolazione"])*1000).fillna(0)
#Average accidents per square kilometer (Accident density)
df_municipality_indicators["Densità_Incidenti_Kmq"] = (df_municipality_indicators["Media Incidenti"]/df_municipality_indicators["Media Superficie(Kmq)"]).fillna(0)

In [42]:
df_municipality_indicators.to_csv("data/municipality_indicators.csv", index=False)

We'll now group the dataset by Region to compute large-scale geographic indicators

In [43]:
df_regional = df_merged.groupby("Regione")[["Incidenti", "Feriti", "Morti", "Superficie(Kmq)","Popolazione_Residente"]].sum().reset_index()

In [44]:
#regional name correction 
regioni = {
    'Trentino A.A.': 'Trentino-Alto Adige/Südtirol',
    'Friuli V.G.': 'Friuli-Venezia Giulia',
    'Emilia Romagna': 'Emilia-Romagna',
    "Valle D'Aosta": "Valle d'Aosta/Vallée d'Aoste"
}
df_aci["Regione"] = df_aci["Regione"].str.replace(regioni)

In [46]:
df_regional_fleet = df_regional.merge(df_aci, on="Regione", how="left")

In [47]:
df_regional_fleet

,Regione,Incidenti,Feriti,Morti,Superficie(Kmq),Popolazione_Residente,Avg_Parco_Auto
0,Abruzzo,33053,48021,816,116430.7260,14160897.0,893945
1,Basilicata,9991,15902,392,108711.6441,6020680.0,379194
2,Calabria,30313,49136,1068,164188.1700,20666810.0,1314156
3,Campania,104813,153975,2456,147902.5204,62483775.0,3564483
4,Emilia-Romagna,180141,239421,3373,246342.7248,48878657.0,2937312
5,Friuli-Venezia Giulia,35443,46598,787,85687.8118,13230772.0,805868
6,Lazio,210098,284745,3630,187906.8295,63115265.0,3844941
7,Liguria,86349,107423,777,58560.0068,16801180.0,841684
8,Lombardia,330246,444935,4540,259514.5930,109660498.0,6213575
9,Marche,55293,76907,980,102518.2981,16586484.0,1036133


In [48]:
#We can now compute regional metrics to analyze the impact of population,territory size, and vehicle fleet density.
df_regional_fleet['Veicoli_Per_Kmq'] = (df_regional_fleet["Avg_Parco_Auto"] / df_regional_fleet['Superficie(Kmq)']) #Vehicle density
df_regional_fleet['Motorizzazione_Pro_Capite_1000'] = ((df_regional_fleet["Avg_Parco_Auto"] / df_regional_fleet["Popolazione_Residente"]) * 1000)
df_regional_fleet['Rapporto Incidenti/Veicoli'] = ((df_regional_fleet["Incidenti"] / df_regional_fleet["Avg_Parco_Auto"]) * 1000) #incident to vehicle ratio per 1k vehicles

In [49]:
df_regional_fleet.to_csv("data/regiona_fleet_metrics.csv",index=False)